In [ ]:
import requests
import pandas as pd
import io

# ==============================================================================
# OECD API CONFIGURATION (OECD Data Explorer - SDMX 2.1 Architecture)
# ==============================================================================
# INSTRUCTIONS TO REPLACE THE INDICATOR (e.g., Public Debt, Inflation):
# 1. Search for the dataset in the new "OECD Data Explorer" portal.
# 2. Above the data table, select the "Developer API" icon.
# 3. Extract the AGENCY_ID and DATAFLOW_ID parameters from the generated URL to replace them below.
# ==============================================================================

# Identifiers for the "Gross domestic spending on R&D" indicator
AGENCY_ID = "OECD.STI.STP"
DATAFLOW_ID = "DSD_MSTI@DF_MSTI"
VERSION = "all"
DIMENSIONS = "all"

# Requested time filters (2000-2025)
START_PERIOD = "2000"
END_PERIOD = "2025"

url = f"https://sdmx.oecd.org/public/rest/data/{AGENCY_ID},{DATAFLOW_ID},{VERSION}/{DIMENSIONS}"

params = {
    "startPeriod": START_PERIOD,
    "endPeriod": END_PERIOD,
    "dimensionAtObservation": "AllDimensions",
    "format": "csvfilewithlabels"
}

headers = {
    "Accept": "application/vnd.sdmx.data+csv; charset=utf-8; version=2",
    "User-Agent": "DataExtractionScript/1.0 (Python/Requests)"
}

# ==============================================================================
# HTTP CALL AND PANDAS PROCESSING
# ==============================================================================

response = requests.get(url, params=params, headers=headers)

if response.status_code == 200:
    csv_stream = io.StringIO(response.text)
    df = pd.read_csv(csv_stream)
    
    # --------------------------------------------------------------------------
    # DATA CLEANING AND TIDY FORMAT CREATION
    # --------------------------------------------------------------------------
    df_clean = df.dropna(subset=['OBS_VALUE']).copy()
    
    target_columns = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
    
    if all(col in df_clean.columns for col in target_columns):
        df_tidy = df_clean[target_columns].rename(columns={
            'REF_AREA': 'Country',
            'TIME_PERIOD': 'Year',
            'OBS_VALUE': 'Value'
        })
        
        df_tidy = df_tidy.reset_index(drop=True)
        
        # ======================================================================
        # FILE SAVING SECTION
        # ======================================================================
        
        # OPTION 1: Save locally in the current Jupyter environment folder
        output_filename = "OECD_RD_Spending_2000_2025.csv"
        df_tidy.to_csv(output_filename, index=False)
        print(f"Extraction completed. File successfully saved as: {output_filename}")
        
        # OPTION 2: If you are using Google Colab and want to save directly to Google Drive
        # Uncomment the following lines to mount your Drive and save it there:
        #
        # from google.colab import drive
        # drive.mount('/content/drive')
        # drive_path = f"/content/drive/MyDrive/{output_filename}"
        # df_tidy.to_csv(drive_path, index=False)
        # print(f"File successfully saved to Google Drive at: {drive_path}")
        
    else:
        print("Structural error: unable to find standard SDMX columns (REF_AREA, TIME_PERIOD, OBS_VALUE).")
        print("Columns currently provided by the API:", df_clean.columns.tolist())
else:
    print(f"Error {response.status_code} during data download.")
    print("Response details:", response.text)

Extraction completed successfully. DataFrame preview:
  Country  Year  Value
0     CHL  2001    0.0
1     CHL  2003    0.0
2     COL  2006    0.0
3     COL  2007    0.0
4     CRI  2001    0.0


C:\Users\3003f\AppData\Local\Temp\ipykernel_960\3899059826.py:50: DtypeWarning: Columns (20,21,22,23,24,25,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_stream)
